# L36 — Model Credibility and Professional Documentation

**Module**: M10 | **Chapter**: 12 | **Lecture**: L36

## Learning Objectives
By the end of this notebook you will be able to:
1. Apply Sargent's V&V framework to structure a credibility argument.
2. Write a complete simulation study report with executive summary, model description, results, and limitations.
3. Communicate confidence intervals and model uncertainty to a non-technical audience.
4. Implement version control practices for reproducible simulation studies.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

A verified and validated model that is poorly documented is not usable in practice.
Credibility is earned not just through technical rigor, but through clear communication.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
import hashlib, json, platform, datetime
from scipy import stats
from dataclasses import dataclass, asdict

## 1. The Sargent V&V Framework

Sargent (2013) describes three objects and three validation activities:

```
Real World ←── Conceptual Model ←── Simulation Model
     ↑                 ↑                   ↑
 Problem Entity   Conceptual         Computer
                  Validity           Model
                                     Verification
         ←──────────────────────────────
                Operational Validity
```

- **Conceptual validity**: Does the conceptual model accurately represent the real system?
- **Verification**: Does the simulation code correctly implement the conceptual model?
- **Operational validity**: Does the simulation output match real system behavior for the intended purpose?

In [ ]:
# Sargent checklist — fill this in for any simulation project
sargent_checklist = {
    'Conceptual Validity': [
        ('Entity/resource table reviewed by domain expert', True),
        ('Assumptions documented and justified', True),
        ('Boundary decision justified relative to study question', True),
        ('Event list complete (no missing state transitions)', True),
    ],
    'Verification': [
        ('Extreme-condition tests pass (zero load, full load)', True),
        ('Single-customer trace matches manual calculation', True),
        ('Flow balance holds (throughput = arrivals - abandonments)', True),
        ('Code reviewed by second analyst', False),  # not yet done
    ],
    'Operational Validity': [
        ('Historical data comparison: RMSE < 2 min', True),
        ('Theil U < 1.0', True),
        ('Sensitivity validation: correct direction for all factors', True),
        ('Predictive validation on holdout data', True),
    ],
}

for category, checks in sargent_checklist.items():
    passed = sum(v for _, v in checks)
    print(f"{category}: {passed}/{len(checks)} checks passed")
    for name, status in checks:
        print(f"  {'✓' if status else '✗'} {name}")
    print()

## 2. Reproducibility Metadata

Every simulation report must record the environment and seed so results can be reproduced exactly.

In [ ]:
@dataclass
class StudyMetadata:
    study_name:    str
    analyst:       str
    date:          str
    python_version: str
    numpy_version: str
    simpy_version: str
    master_seed:   int
    n_replications: int
    data_checksum: str


def compute_data_checksum(filepath: str) -> str:
    """SHA-256 checksum of a data file for reproducibility."""
    try:
        with open(filepath, 'rb') as f:
            return hashlib.sha256(f.read()).hexdigest()[:16]
    except FileNotFoundError:
        return 'file-not-found'


meta = StudyMetadata(
    study_name    = 'Primary Care Clinic Staffing Study',
    analyst       = 'iLab.Utk',
    date          = datetime.date.today().isoformat(),
    python_version = platform.python_version(),
    numpy_version  = np.__version__,
    simpy_version  = simpy.__version__,
    master_seed   = 42,
    n_replications = 30,
    data_checksum  = compute_data_checksum('../../../data/clinic_arrivals.csv'),
)

print(json.dumps(asdict(meta), indent=2))

## 3. Complete Simulation Study Results

We run the full study: compare 3 staffing levels, 30 replications each.

In [ ]:
def run_clinic_full(n_nurses, seed, n_patients=500, warmup=100):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    reg   = simpy.Resource(env, capacity=1)
    nurse = simpy.Resource(env, capacity=n_nurses)
    sojourns, nurse_busy = [], []

    def patient():
        t0 = env.now
        with reg.request() as req:
            yield req
            yield env.timeout(rng.exponential(3.0))   # 3 min registration
        with nurse.request() as req:
            yield req
            svc = rng.exponential(8.0)                # 8 min triage
            nurse_busy.append(svc)
            yield env.timeout(svc)
        sojourns.append(env.now - t0)

    def arrivals():
        for _ in range(n_patients):
            env.process(patient())
            yield env.timeout(rng.exponential(12.0))  # 12 min between arrivals

    env.process(arrivals())
    env.run()
    arr = sojourns[warmup:]
    T = env.now
    return {
        'W':    np.mean(arr),
        'W_p90': np.percentile(arr, 90),
        'rho':  sum(nurse_busy) / (n_nurses * T) if T > 0 else 0,
    }


N_REPS = 30
staffing_levels = [1, 2, 3]
results = {}

master_rng = np.random.default_rng(meta.master_seed)
for n in staffing_levels:
    seeds = master_rng.integers(0, 10**6, size=N_REPS)
    reps = [run_clinic_full(n, s) for s in seeds]
    W_arr   = np.array([r['W'] for r in reps])
    rho_arr = np.array([r['rho'] for r in reps])
    h = stats.t.ppf(0.975, N_REPS-1) * W_arr.std(ddof=1) / np.sqrt(N_REPS)
    results[n] = {
        'W_mean':  W_arr.mean(),
        'W_ci_lo': W_arr.mean() - h,
        'W_ci_hi': W_arr.mean() + h,
        'rho':     rho_arr.mean(),
        'W_reps':  W_arr,
    }
    print(f"n_nurses={n}: W={W_arr.mean():.1f} [{W_arr.mean()-h:.1f},{W_arr.mean()+h:.1f}] min  ρ={rho_arr.mean():.3f}")

## 4. Communicating Uncertainty to Decision Makers

Confidence intervals are often misunderstood. A good report translates statistical output into actionable language.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# CI plot
for n in staffing_levels:
    r = results[n]
    ax1.errorbar(n, r['W_mean'],
                 yerr=[[r['W_mean'] - r['W_ci_lo']], [r['W_ci_hi'] - r['W_mean']]],
                 fmt='o', capsize=8, color='steelblue', ms=8)
ax1.set_xlabel('Number of triage nurses')
ax1.set_ylabel('Mean sojourn time W (min)')
ax1.set_title(f'Mean patient time in clinic with 95% CI ({N_REPS} reps)')
ax1.set_xticks(staffing_levels)
ax1.axhline(30, color='red', lw=1.5, linestyle='--', label='Target: W ≤ 30 min')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Utilisation
ax2.bar(staffing_levels, [results[n]['rho'] for n in staffing_levels],
        color='steelblue', alpha=0.7)
ax2.axhline(0.70, color='orange', lw=1.5, linestyle='--', label='Min util. 70%')
ax2.axhline(1.00, color='red',    lw=1.5, linestyle=':', label='Stability limit')
ax2.set_xlabel('Number of triage nurses')
ax2.set_ylabel('Nurse utilisation ρ')
ax2.set_title('Nurse utilisation by staffing level')
ax2.set_xticks(staffing_levels)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Executive Summary (Decision-Maker Format)

The executive summary should convey the recommendation without requiring the reader to understand confidence intervals, p-values, or simulation mechanics.

In [ ]:
best_n = min(staffing_levels,
             key=lambda n: results[n]['W_mean'] if results[n]['rho'] >= 0.60 else float('inf'))

exec_summary = f"""
EXECUTIVE SUMMARY
Primary Care Clinic — Triage Staffing Study
Date: {meta.date}

QUESTION: How many triage nurses minimise patient waiting time
          while keeping nurse utilisation above 70%?

RECOMMENDATION: {best_n} triage nurse(s)

RESULTS:
  • With {best_n} nurse(s), patients spend an average of {results[best_n]['W_mean']:.0f} minutes
    in the clinic from arrival to end of triage.
  • Our confidence in this estimate: the true average is between
    {results[best_n]['W_ci_lo']:.0f} and {results[best_n]['W_ci_hi']:.0f} minutes (19 times out of 20).
  • Nurse utilisation is {results[best_n]['rho']*100:.0f}%, meeting the 70% operational target.

WHAT WOULD CHANGE:
  • If patient arrivals increase by 20%, an additional nurse may be needed.
  • These results assume constant arrival rates throughout the day.
    Morning peaks may require temporary surge staffing.

CONFIDENCE IN THE MODEL:
  • The model was verified against analytical formulas and historical records.
  • Model outputs match 30 days of observed data within 2 minutes (RMSE).
  • We recommend revisiting this analysis if arrival patterns change.
"""
print(exec_summary)

## 6. Limitations Section

Every professional report must explicitly state what the model cannot answer.

In [ ]:
limitations = [
    ("Constant arrival rate",
     "The model uses a constant λ=5/hr. Real clinics have morning/afternoon peaks. "
     "If peak rates exceed 7/hr, the recommended staffing may be insufficient."),

    ("Exponential service times",
     "Triage service times are modelled as exponential (memoryless). If the actual "
     "distribution has a heavier tail, Wq could be higher by up to 30%."),

    ("No patient abandonment",
     "The model assumes unlimited patience. In practice, some patients leave after "
     "waiting 45+ minutes. This would reduce observed Wq but also reduce effective "
     "throughput."),

    ("Steady-state assumption",
     "Results apply to the steady-state operating period. The model does not capture "
     "the opening surge (first 30 minutes of the clinic day)."),

    ("No downstream effects",
     "The model stops at end of triage. Delays in the exam room could back-pressure "
     "the triage queue in ways the model does not capture."),
]

print("MODEL LIMITATIONS")
print("=" * 60)
for i, (title, desc) in enumerate(limitations, 1):
    print(f"\n{i}. {title}")
    print(f"   {desc}")

---
## Try It Yourself

1. **Decision-maker Q&A**: Write responses to the following questions a clinic manager might ask:
   - "What does '95% confidence interval' mean in plain English?"
   - "If the model shows 2 nurses is best, why can't we just trust that answer without all these statistics?"
   - "Our clinic has some special cases — patients who need two nurses simultaneously. Can the model handle that?"

2. **Reproducibility check**: Modify the study metadata to use a different `master_seed`. Re-run the study. Do the same staffing level win? Is the CI similar in width? How many seeds would you need to check before being confident the recommendation is seed-independent?

3. **Version control practice**: The study metadata records `data_checksum`. Explain why this is important. What would happen to the study's reproducibility if `clinic_arrivals.csv` were updated with new observations but the old report was still in circulation? How should a simulation analyst handle dataset versioning?